In [1]:
print("vLLM 测试开始，请确保已下载模型到缓存目录！")

vLLM 测试开始，请确保已下载模型到缓存目录！


In [2]:
import os
# 必须在 import vllm 之前设置
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_OFFLINE"] = "1"   # 离线模式，仅使用缓存
from vllm import LLM
from vllm import SamplingParams
import torch
import time

INFO 07-23 11:11:49 __init__.py:190] Automatically detected platform cuda.


In [ ]:
def benchmark_llm(model_name, gpu_memory_utilization, max_model_len, enforce_eager, warmup=2, test=5):
    """
    加载模型并测试生成速度和显存占用
    返回：平均生成时间、显存占用
    """
    print(f"\n{'='*60}")
    print(f"正在测试: {model_name}")
    print(f"参数: gpu_memory_utilization={gpu_memory_utilization}, "
          f"max_model_len={max_model_len}, enforce_eager={enforce_eager}")
    print(f"{'='*60}")

    llm = LLM(
        model=model_name,
        gpu_memory_utilization=gpu_memory_utilization,
        max_model_len=max_model_len,
        enforce_eager=enforce_eager,
    )
      # 测试 prompt
    prompts = [
        "请详细解释什么是深度学习？",
        "Write a short story about a robot learning to paint.",
        "列出五种常见的 Python 优化技巧，并简要说明。"
    ]

    # 采样参数
    sampling_params = SamplingParams(
        temperature=0.8,
        top_p=0.95,
        max_tokens=256
    )

    # 预热
    print("预热中...")
    for i in range(warmup):
        _ = llm.generate(prompts[:1], sampling_params)

    # 计时测试
    print(f"测试中（{test} 轮）...")
    start_time = time.time()
    for i in range(test):
        outputs = llm.generate(prompts, sampling_params)
    end_time = time.time()

    avg_time = (end_time - start_time) / test
    print(f"平均生成时间: {avg_time:.2f} 秒")

    # 显存占用
    if torch.cuda.is_available():
        memory_allocated = torch.cuda.max_memory_allocated() / 1024**3
        memory_reserved = torch.cuda.max_memory_reserved() / 1024**3
        print(f"PyTorch 最大分配显存: {memory_allocated:.2f} GiB")
        print(f"PyTorch 最大预留显存: {memory_reserved:.2f} GiB")
        torch.cuda.reset_peak_memory_stats()  # 重置统计

    # 清理
    del llm
    torch.cuda.empty_cache()

    return avg_time, memory_allocated, memory_reserved

In [4]:

# 定义测试矩阵
test_configs = [
    # (模型名, GPU 内存利用率, 最大序列长度, 是否禁用 CUDA Graph)
    ("facebook/opt-125m", 0.5, 512, True),    # 基础对照组
    ("Qwen/Qwen2.5-1.5B-Instruct", 0.5, 512, True),  # 你的主力模型
    ("Qwen/Qwen2.5-1.5B-Instruct", 0.6, 1024, True), # 更大内存限制
    ("Qwen/Qwen2.5-1.5B-Instruct", 0.5, 512, False), # 启用 CUDA Graph
]

results = []
for model, gpu_mem, max_len, eager in test_configs:
    try:
        avg_time, memory_allocated, memory_reserved = benchmark_llm(model, gpu_mem, max_len, eager)
        results.append({
            "model": model,
            "gpu_memory_utilization": gpu_mem,
            "max_model_len": max_len,
            "enforce_eager": eager,
            "avg_time": avg_time,
            "memory_allocated": memory_allocated,
            "memory_reserved": memory_reserved
        })
    except Exception as e:
        print(f"测试失败: {e}")


正在测试: facebook/opt-125m
参数: gpu_memory_utilization=0.5, max_model_len=512, enforce_eager=True
INFO 07-23 11:11:57 config.py:542] This model supports multiple tasks: {'classify', 'reward', 'embed', 'generate', 'score'}. Defaulting to 'generate'.
WARNING 07-23 11:11:57 cuda.py:95] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
WARNING 07-23 11:11:57 config.py:678] Async output processing is not supported on the current platform type cuda.
INFO 07-23 11:11:57 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=512, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_r

Loading pt checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 07-23 11:12:04 model_runner.py:1115] Loading model weights took 0.2389 GB
INFO 07-23 11:12:05 worker.py:267] Memory profiling takes 0.55 seconds
INFO 07-23 11:12:05 worker.py:267] the current vLLM instance can use total_gpu_memory (11.76GiB) x gpu_memory_utilization (0.50) = 5.88GiB
INFO 07-23 11:12:05 worker.py:267] model weights take 0.24GiB; non_torch_memory takes -0.01GiB; PyTorch activation peak memory takes 0.47GiB; the rest of the memory reserved for KV Cache is 5.18GiB.
INFO 07-23 11:12:05 executor_base.py:110] # CUDA blocks: 9432, # CPU blocks: 7281
INFO 07-23 11:12:05 executor_base.py:115] Maximum concurrency for 512 tokens per request: 294.75x
INFO 07-23 11:12:08 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 3.59 seconds
预热中...


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.70s/it, est. speed input: 17.68 toks/s, output: 150.84 toks/s]


测试中（5 轮）...


...

预热中...


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.82s/it, est. speed input: 3.85 toks/s, output: 140.79 toks/s]


测试中（5 轮）...


Processed prompts: 100%|██████████| 3/3 [00:02<00:00,  1.43it/s, est. speed input: 14.75 toks/s, output: 331.73 toks/s]

平均生成时间: 2.06 秒
PyTorch 最大分配显存: 4.47 GiB
PyTorch 最大预留显存: 4.69 GiB


In [5]:
print(
    f"| {'model':^35} | {'GPU(%)':^10} | {'MaxLength':^10} | {'Eager':^8} | "
    f"{'AvgTimes(s)':^12} | {'AllocatedMemory(GiB)':^20} | {'ReservedMemory(GiB)':^20} |"
)
print("|","-" * 133, "|")

for r in results:
    line = (
        f"| {r['model']:^35} | {r['gpu_memory_utilization']:^10.1f} | "
        f"{r['max_model_len']:^10} | {str(r['enforce_eager']):^8} | "
        f"{r['avg_time']:^12.2f} | {r['memory_allocated']:^20.2f} | {r['memory_reserved']:^20.2f} |"
    )
    print(line)

print("\n实验完成！请记录以上数据，思考以下问题：")
print("1. gpu_memory_utilization 如何影响显存占用和速度？")
print("2. enforce_eager=True/False 对性能有何影响？")
print("3. 不同模型对参数的敏感度有何差异？")

|                model                |   GPU(%)   | MaxLength  |  Eager   | AvgTimes(s)  | AllocatedMemory(GiB) | ReservedMemory(GiB)  |
| ------------------------------------------------------------------------------------------------------------------------------------- |
|          facebook/opt-125m          |    0.5     |    512     |   True   |     1.80     |         5.44         |         5.48         |
|     Qwen/Qwen2.5-1.5B-Instruct      |    0.5     |    512     |   True   |     4.41     |         4.47         |         4.64         |
|     Qwen/Qwen2.5-1.5B-Instruct      |    0.6     |    1024    |   True   |     4.35     |         5.67         |         5.82         |
|     Qwen/Qwen2.5-1.5B-Instruct      |    0.5     |    512     |  False   |     2.06     |         4.47         |         4.69         |

实验完成！请记录以上数据，思考以下问题：
1. gpu_memory_utilization 如何影响显存占用和速度？
2. enforce_eager=True/False 对性能有何影响？
3. 不同模型对参数的敏感度有何差异？
